Import Packages

In [87]:
# !pip install statsbombpy
# !pip install category_encoders

In [88]:
import numpy as np
import pandas as pd
from statsbombpy import sb
import pandas as pd
import category_encoders as ce
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression

In [89]:
# Import matches and events
pairs = [(72, 30), (72, 107)]

matches = pd.concat(
    [sb.matches(competition_id=c, season_id=s) for c, s in pairs],
    ignore_index=True,
)

events = [sb.events(m) for m in matches.match_id]


/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: 

In [90]:
# Combine all event DataFrames into a single DataFrame
events_df = pd.concat(events, ignore_index=True)

# Extract shots from events
shots = events_df[events_df["type"] == "Shot"].copy()

# Remove penalties
shots = shots[shots["shot_type"] != "Penalty"].copy()

# Remove target
shots["goal"] = (shots["shot_outcome"] == "Goal").astype(int)

Stats bomb measures the field as 120x80, therefore subtracting 120 will determine the `x` distance from the goal. We are subtracting 40 from `y` since the goal sits in the center of the y axis and we want the distance from the goal.

In [91]:
def calc_angle_dist(X):
    x, y = X["x"], X["y"]
    distance = np.hypot(120 - x, 40 - y)
    a_x, a_y = 120 - x, 36 - y
    b_x, b_y = 120 - x, 44 - y
    cos = (a_x*b_x + a_y*b_y) / (np.hypot(a_x, a_y) * np.hypot(b_x, b_y))
    angle = np.arccos(np.clip(cos, -1, 1))
    return pd.DataFrame({"distance": distance, "angle": angle}, index=X.index)

In [92]:
shots["x"] = shots["location"].str[0]
shots["y"] = shots["location"].str[1]

In [93]:
sorted(shots['shot_outcome'].unique())

['Blocked',
 'Goal',
 'Off T',
 'Post',
 'Saved',
 'Saved Off Target',
 'Saved to Post',
 'Wayward']

In [97]:
# Calculate distance and angle
shots[["distance", "angle"]] = calc_angle_dist(shots)

preprocess = ColumnTransformer([
    ("body", OneHotEncoder(drop=["Right Foot"], sparse_output=False, handle_unknown="ignore"), ["shot_body_part"]), # Right Foot is Baseline
    ("type", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), ["shot_type"]), # Corner is Baseline
    ("num", StandardScaler(), ["x", "y", "distance", "angle"]),
], remainder="passthrough")

In [98]:
flags = ["under_pressure", "shot_one_on_one", "shot_follows_dribble", "shot_first_time"]
shots[flags] = shots[flags].fillna(False).astype(int)

In [106]:
# Define columns in DF
cols = ["x", "y", "distance", "angle", "shot_body_part", "shot_type",
        "under_pressure", "shot_one_on_one", "shot_follows_dribble", "shot_first_time"]
preprocess.set_output(transform="pandas")
X_prep = preprocess.fit_transform(shots[cols])
X_prep.columns

Index(['body__shot_body_part_Head', 'body__shot_body_part_Left Foot',
       'body__shot_body_part_Other', 'type__shot_type_Free Kick',
       'type__shot_type_Kick Off', 'type__shot_type_Open Play', 'num__x',
       'num__y', 'num__distance', 'num__angle', 'remainder__under_pressure',
       'remainder__shot_one_on_one', 'remainder__shot_follows_dribble',
       'remainder__shot_first_time'],
      dtype='object')